# Dashboard: Live Status Monitor

**Purpose:** Subscribe to all MQTT topics and display real-time aggregated status.

**Subscribes to:** All topics under `city/flood/#`

**Display:** Sensor count, average water level, current alert status

**Update Interval:** Every 2 seconds

In [9]:
import json
import time
from datetime import datetime, timezone
from IPython.display import display

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector
from simulated_city.flood import ObserverReading, ResponseStatus

try:
    from simulated_city.maplibre_live import LiveMapLibreMap as MapWidget
except Exception:
    from anymap_ts import Map as MapWidget

cfg = load_config()
mqtt_cfg = cfg.mqtt

SAFE_LAT, SAFE_LON = 55.456553861769855, 12.181774944848945      # Køge Torv
FLOOD_LAT, FLOOD_LON = 55.45058843620187, 12.197503222443261    # Køge Søndre Strand

sensor_readings = {}
control_status = {"alert": "low", "timestamp": None}
pedestrians = {}
marker_ids = set()
update_count = 0

map_widget = MapWidget(center=(FLOOD_LON, FLOOD_LAT), zoom=14, height="550px", width="100%")
try:
    map_widget.add_basemap("OpenStreetMap.Mapnik")
except Exception:
    pass

map_widget.add_marker(SAFE_LON, SAFE_LAT, name="zone-safe", color="#2e7d32", popup="Køge Torv (safe zone)")
map_widget.add_marker(FLOOD_LON, FLOOD_LAT, name="zone-flood", color="#c62828", popup="Køge Søndre Strand (flood zone)")
marker_ids.update({"zone-safe", "zone-flood"})

display(map_widget)
print("Dashboard configured for Køge flood simulation")
print(f"Base topic: {mqtt_cfg.base_topic}")


from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector

cfg = load_config()
print(f"Config loaded: {cfg.mqtt.host}:{cfg.mqtt.port}")

mqtt = MqttConnector(cfg.mqtt, client_id_suffix="test")
try:
    mqtt.connect()
    connected = mqtt.wait_for_connection(timeout=5)
    print(f"MQTT connection: {'✓ OK' if connected else '✗ Failed'}")
    mqtt.disconnect()
except Exception as e:
    print(f"✗ Error: {e}")

Dashboard configured for Køge flood simulation
Base topic: simulated-city
Config loaded: 127.0.0.1:1883
MQTT connection: ✓ OK


Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


In [2]:
mqtt = MqttConnector(mqtt_cfg, client_id_suffix="dashboard")
mqtt.connect()

if mqtt.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    raise RuntimeError("Failed to connect to MQTT broker")


def _upsert_person_marker(person):
    marker_id = f"ped-{person['id']}"
    color = "#1565c0" if person.get("moving") else "#43a047"
    popup = f"{person['id']} | {person.get('location', 'unknown')}"

    try:
        if hasattr(map_widget, "move_marker") and marker_id in marker_ids:
            map_widget.move_marker(marker_id, (person["lon"], person["lat"]), color=color, popup=popup)
        else:
            if marker_id in marker_ids:
                map_widget.remove_marker(marker_id)
            map_widget.add_marker(person["lon"], person["lat"], name=marker_id, color=color, popup=popup)
            marker_ids.add(marker_id)
    except Exception:
        # Never crash the dashboard loop on UI update errors.
        pass


def on_message(client, userdata, msg):
    topic = msg.topic
    try:
        data = json.loads(msg.payload.decode())
    except Exception:
        return

    if "/observer/" in topic:
        try:
            reading = ObserverReading.from_json(data)
            sensor_readings[reading.sensor_id] = (reading.water_level, reading.timestamp)
        except Exception:
            pass
    elif "/control/command" in topic:
        severity = data.get("parameters", {}).get("severity", "low")
        control_status["alert"] = severity
        control_status["timestamp"] = data.get("timestamp")
    elif "/response/status" in topic:
        try:
            status = ResponseStatus.from_json(data)
            details = status.details if isinstance(status.details, dict) else {}
            for person in details.get("pedestrians", []):
                pedestrians[person["id"]] = person
                _upsert_person_marker(person)
        except Exception:
            pass


mqtt.client.on_message = on_message
mqtt.client.subscribe(f"{mqtt_cfg.base_topic}/#")
print(f"✓ Subscribed to: {mqtt_cfg.base_topic}/#")

✓ Connected to MQTT broker
✓ Subscribed to: simulated-city/#


In [3]:
from simulated_city.geo import distance_wgs84


def display_status():
    global update_count
    update_count += 1

    if sensor_readings:
        avg_level = sum(level for level, _ in sensor_readings.values()) / len(sensor_readings)
        sensor_count = len(sensor_readings)
    else:
        avg_level = 0.0
        sensor_count = 0

    alert = control_status.get("alert", "low")
    indicator = "🔴 HIGH" if alert == "high" else "🟢 LOW"

    evac_distance_m = distance_wgs84(SAFE_LAT, SAFE_LON, FLOOD_LAT, FLOOD_LON)
    evac_distance_km = evac_distance_m / 1000

    moving_count = sum(1 for p in pedestrians.values() if p.get("moving"))

    print(f"[{update_count}] SENSORS: {sensor_count} active  |  Avg water: {avg_level:.2f}m")
    print(f"         ALERT: {indicator}  |  Evac distance: {evac_distance_km:.2f}km")
    print(f"         Pedestrians on map: {len(pedestrians)}  |  Moving: {moving_count}")

In [8]:
print("\n" + "=" * 64)
print("LIVE STATUS - Køge Flood Simulation (Map + MQTT)")
print("=" * 64)
print("(Press Ctrl+C to stop)\n")

try:
    while True:
        display_status()
        time.sleep(2)
except KeyboardInterrupt:
    print("\n\n✓ Dashboard stopped by user")
finally:
    mqtt.disconnect()


LIVE STATUS - Køge Flood Simulation (Map + MQTT)
(Press Ctrl+C to stop)

[88] SENSORS: 5 active  |  Avg water: 0.07m
         ALERT: 🟢 LOW  |  Evac distance: 1.19km
         Pedestrians on map: 0  |  Moving: 0
[89] SENSORS: 5 active  |  Avg water: 0.07m
         ALERT: 🟢 LOW  |  Evac distance: 1.19km
         Pedestrians on map: 0  |  Moving: 0
[90] SENSORS: 5 active  |  Avg water: 0.07m
         ALERT: 🟢 LOW  |  Evac distance: 1.19km
         Pedestrians on map: 0  |  Moving: 0
[91] SENSORS: 5 active  |  Avg water: 0.07m
         ALERT: 🟢 LOW  |  Evac distance: 1.19km
         Pedestrians on map: 0  |  Moving: 0
[92] SENSORS: 5 active  |  Avg water: 0.07m
         ALERT: 🟢 LOW  |  Evac distance: 1.19km
         Pedestrians on map: 0  |  Moving: 0
[93] SENSORS: 5 active  |  Avg water: 0.07m
         ALERT: 🟢 LOW  |  Evac distance: 1.19km
         Pedestrians on map: 0  |  Moving: 0
[94] SENSORS: 5 active  |  Avg water: 0.07m
         ALERT: 🟢 LOW  |  Evac distance: 1.19km
         Ped

In [5]:
print("This legacy display cell is disabled. Use the active loop cell above.")

This legacy display cell is disabled. Use the active loop cell above.


## Live Status Display

Real-time stats from MQTT streams (auto-updates while running)

In [6]:
print("This legacy MQTT setup cell is disabled. Use the active setup/subscription cells above.")

This legacy MQTT setup cell is disabled. Use the active setup/subscription cells above.


## Connect to MQTT and Subscribe to All Streams

In [7]:
print("This legacy configuration cell is disabled. Use the active configuration cell above.")

This legacy configuration cell is disabled. Use the active configuration cell above.


## Setup and Configuration

# Dashboard: Live Flood Simulation Monitor

**Real-time visualization of the flood simulation using anymap-ts.**

Displays:
- **Sensors**: Blue markers at Køge Søndre Strand with water level
- **Pedestrians**: Green markers at current position (strand or torv)
- **Alert Status**: Color-coded (green = safe, red = danger)
- **Water Level**: Displayed in status bar

All data streams from MQTT brokers in real-time.

# Dashboard Notebook
Subscribes to all MQTT topics and renders a live map using `anymap-ts`.
Used for visualizing the flood simulation.